In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch06-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "06-03-maccs-keys.py"

# 乙酸、乙酸根與乙酸鈉的 MACCS keys
先用少數結構追蹤位元，再載入 ESOL 檢查同一個問題。首次執行會下載 ESOL。如要增寫練習程式，可接在分析儲存格的末尾，再重跑該格與最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, MACCSkeys, rdFingerprintGenerator

np.random.seed(42)
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)

# 概念示意 — 實際應用請使用下方 DeepChem 版本
examples = {"Acetic acid": "CC(=O)O", "Acetate": "CC(=O)[O-]",
            "Sodium acetate": "CC(=O)[O-].[Na+]"}
keys = [49, 139, 154, 159, 164, 166]
mols = [Chem.MolFromSmiles(smi) for smi in examples.values()]
fps = [MACCSkeys.GenMACCSKeys(mol) for mol in mols]
for name, mol, fp in zip(examples, mols, fps):
    print(name, fp.GetNumBits(), len(Chem.GetMolFrags(mol)), list(fp.GetOnBits()))
matrix = [[int(fp[key]) for key in keys] for fp in fps]
fig = plt.figure(figsize=(9, 4.8), layout="constrained")
grid = fig.add_gridspec(2, 1, height_ratios=[1, 1.15])
structures = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(275, 145),
                                    legends=list(examples))
top = fig.add_subplot(grid[0])
top.imshow(structures)
top.axis("off")
ax = fig.add_subplot(grid[1])
from matplotlib.colors import ListedColormap
ax.imshow(matrix, aspect="auto", cmap=ListedColormap(["#edf1f4", "#126658"]))
ax.set_xticks(range(len(keys)), [f"Key {k}" for k in keys])
ax.set_yticks(range(len(mols)), list(examples))
for i, row in enumerate(matrix):
    for j, bit in enumerate(row):
        ax.text(j, i, str(bit), ha="center", va="center", color="white" if bit else "#293b45")
fig.savefig(figure_dir / "06-03-maccs-keys.png", dpi=160)

# ESOL：1,128 筆實測水溶解度 log10(mol/L)，原始研究 DOI 10.1021/ci034243x。
# 只描述輸入結構，不訓練模型或轉換標籤。
tasks, splits, _ = dc.molnet.load_delaney(featurizer=dc.feat.RawFeaturizer(smiles=True),
    splitter="scaffold", transformers=[], data_dir=str(cache), reload=False)
esol_mols = [Chem.MolFromSmiles(smi) for split in splits for smi in split.ids]
assert len(esol_mols) == 1128 and all(mol is not None for mol in esol_mols)
esol_fps = [MACCSkeys.GenMACCSKeys(mol) for mol in esol_mols]
for key in keys:
    print(f"Key {key}: {sum(fp[key] for fp in esol_fps)}/{len(esol_fps)}")
assert all(fp[166] == int(len(Chem.GetMolFrags(mol)) > 1) for mol, fp in zip(esol_mols, esol_fps))

In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "06-03-maccs-keys.png")))